# 📘 Week 3 · Day 15-16: 검증 전략 (CV) & 평가 지표

## 🎯 학습 목표
- **Trust your CV, not LB(leaderboard)** 원칙 이해
- 상황별 올바른 **Cross-Validation 전략** 선택
- 모든 **평가 지표**의 의미·직접 구현·함정
- **로컬 CV와 LB가 다를 때** 어떻게 대응할지

> 🔥 Kaggle 우승자 인터뷰에서 가장 많이 나오는 말: **"CV가 안정적이었다."**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    GroupKFold, TimeSeriesSplit, cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
import warnings; warnings.filterwarnings('ignore')
np.random.seed(0)

---

## 1. 왜 단순 train_test_split은 부족한가?

### 문제점
- 한 번의 분할 결과는 **운** 에 영향받음
- 데이터 적을수록 분산 큼
- 하이퍼파라미터 튜닝 시 val set에 overfit

### 해결: Cross-Validation
K번 다른 분할로 평가 → **평균과 표준편차**로 모델 성능 추정.

In [ ]:
# 예시: 동일 모델도 split마다 다른 점수
X, y = make_classification(n_samples=500, random_state=0)

scores_single = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed)
    m = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    scores_single.append(m.score(X_te, y_te))

print(f"10개 다른 split 점수: {np.round(scores_single, 3)}")
print(f"평균 {np.mean(scores_single):.4f} ± {np.std(scores_single):.4f}")

---

## 2. KFold - 가장 기본

데이터를 K개 fold로 나누고, 각 fold를 한 번씩 val로 사용.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)

scores = []
for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    m = LogisticRegression(max_iter=1000).fit(X[tr_idx], y[tr_idx])
    s = m.score(X[val_idx], y[val_idx])
    scores.append(s)
    print(f"Fold {fold+1}: {s:.4f}")

print(f"\nMean: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

### KFold 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
n_samples = 50
kf_viz = KFold(n_splits=5, shuffle=True, random_state=0)
for fold, (tr_idx, val_idx) in enumerate(kf_viz.split(np.arange(n_samples))):
    color = np.array(['lightblue'] * n_samples)
    color[val_idx] = 'salmon'
    ax.scatter(np.arange(n_samples), [fold]*n_samples, c=color, s=50, marker='s')
ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_title('KFold: 파랑=Train, 빨강=Validation')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

---

## 3. StratifiedKFold — 분류 문제 **기본값**

타깃 비율을 각 fold에 **동일하게** 유지. 불균형 데이터에서 필수.

In [ ]:
# 불균형 데이터 생성 (95:5)
from sklearn.datasets import make_classification
X_imb, y_imb = make_classification(n_samples=1000, weights=[0.95, 0.05], random_state=0)
print(f"전체 y=1 비율: {y_imb.mean():.3f}")

# KFold vs StratifiedKFold 비교
print("\n[KFold]")
for fold, (tr, val) in enumerate(KFold(5, shuffle=True, random_state=0).split(X_imb)):
    print(f"  Fold {fold+1} val y=1 ratio: {y_imb[val].mean():.3f}")

print("\n[StratifiedKFold]")
for fold, (tr, val) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(X_imb, y_imb)):
    print(f"  Fold {fold+1} val y=1 ratio: {y_imb[val].mean():.3f}")

> 💡 Stratified에서는 모든 fold의 타깃 비율이 거의 동일 → **신뢰할 수 있는 CV 점수**.

---

## 4. GroupKFold — 같은 그룹은 **절대** 분리

In [ ]:
# 예: 환자 데이터. 한 환자가 여러 날 방문 → 같은 환자가 train/val에 모두 있으면 누설
from sklearn.model_selection import GroupKFold

patients = np.repeat(np.arange(100), 5)  # 100명 × 5회 방문
X_g = np.random.randn(500, 4)
y_g = np.random.randint(0, 2, 500)

gkf = GroupKFold(n_splits=5)
for fold, (tr, val) in enumerate(gkf.split(X_g, y_g, groups=patients)):
    tr_p = set(patients[tr])
    val_p = set(patients[val])
    print(f"Fold {fold+1}: train patients={len(tr_p)}, val patients={len(val_p)}, 겹침={len(tr_p & val_p)}")

> 💡 **언제 쓰나**: 환자 ID, 사용자 ID, 날짜 단위, 이미지의 원본 파일 ID 등.

---

## 5. TimeSeriesSplit — 시계열 필수

**과거로 학습, 미래로 검증**. 미래 누설 금지.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
n = 100
X_t = np.arange(n).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(10, 3))
for fold, (tr, val) in enumerate(tscv.split(X_t)):
    color = np.array(['lightgray'] * n)
    color[tr] = 'steelblue'
    color[val] = 'salmon'
    ax.scatter(np.arange(n), [fold]*n, c=color, s=30, marker='s')
ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_title('TimeSeriesSplit: 회색=미사용, 파랑=Train, 빨강=Val')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

---

## 6. Adversarial Validation — Train/Test 분포가 다를 때

**아이디어**: train에 label=0, test에 label=1을 달고 분류. AUC가 0.5 근처면 OK, 1에 가까우면 분포가 매우 다름.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 시뮬레이션: test가 train보다 평균이 shift됨
X_train = np.random.randn(1000, 5)
X_test  = np.random.randn(500, 5) + 0.8   # 평균 다름

X_all = np.vstack([X_train, X_test])
y_all = np.hstack([np.zeros(1000), np.ones(500)])

X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_all, test_size=0.3, random_state=0)
m = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
print(f"Adversarial AUC = {auc:.4f}")
print("→ 0.5 근처면 분포 동일, 1에 가까우면 분포 다름 (CV 전략 재검토 필요)")

> 🔑 AUC가 높다면:
> - 분포 다른 피처 제거
> - test 쪽에 가까운 샘플만으로 validation
> - 피처 변환 재검토

---

## 7. 평가 지표 완전 정복

### 7.1 분류 지표

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, log_loss, confusion_matrix)

y_true = np.array([0,0,0,0,0,0,0,1,1,1])
y_pred = np.array([0,0,0,0,1,1,0,1,1,0])
y_prob = np.array([.1,.2,.15,.25,.6,.7,.4,.9,.85,.45])

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
print(f"F1:        {f1_score(y_true, y_pred):.3f}")
print(f"AUC:       {roc_auc_score(y_true, y_prob):.3f}")
print(f"LogLoss:   {log_loss(y_true, y_prob):.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

### 7.2 지표 선택 가이드

| 상황 | 지표 | 이유 |
|------|------|------|
| 균형 데이터 | Accuracy | 단순 |
| 불균형 + 둘 다 중요 | **F1** | P와 R 조화평균 |
| 순위가 중요 | **AUC** | 확률 기반 순위 품질 |
| 정확한 확률 필요 | **LogLoss** | 잘못된 확신에 큰 페널티 |
| 암 진단 (놓치면 안됨) | **Recall** 우선 | FN 최소화 |
| 스팸 분류 (오탐 최소) | **Precision** 우선 | FP 최소화 |


### 7.3 회귀 지표

In [ ]:
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, mean_squared_log_error)

y_true = np.array([100, 200, 300, 400, 500])
y_pred = np.array([110, 190, 330, 390, 480])

mse  = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)
mape = (np.abs((y_true - y_pred) / y_true)).mean() * 100
rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred))

print(f"MSE   : {mse:.2f}")
print(f"RMSE  : {rmse:.2f}")
print(f"MAE   : {mae:.2f}")
print(f"R²    : {r2:.4f}")
print(f"MAPE  : {mape:.2f}%")
print(f"RMSLE : {rmsle:.4f}")

### 7.4 회귀 지표 선택 가이드

| 지표 | 특징 | 쓰임 |
|------|------|------|
| **MSE/RMSE** | 큰 오차에 강한 페널티 | 일반적 |
| **MAE** | 이상치 영향 적음 | 중앙값 예측 |
| **RMSLE** | log scale, **비대칭** (과소예측에 더 큰 페널티) | House Prices, 매출 예측 |
| **MAPE** | 상대 오차 | 스케일 다른 항목 비교 |
| **R²** | 설명력 (1 최고) | 해석용, 순위 결정엔 부적합 |

---

## 8. ROC Curve와 PR Curve

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

y_true = np.hstack([np.zeros(500), np.ones(100)])  # 불균형 데이터
y_prob = np.hstack([np.random.beta(2, 5, 500), np.random.beta(5, 2, 100)])

fpr, tpr, _ = roc_curve(y_true, y_prob)
prec, rec, _ = precision_recall_curve(y_true, y_prob)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0,1],[0,1],'k--')
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].set_title(f'ROC (AUC={roc_auc_score(y_true,y_prob):.3f})')

ax[1].plot(rec, prec)
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].set_title('Precision-Recall')
plt.tight_layout(); plt.show()

> 💡 **불균형 데이터**에선 ROC보다 **PR 커브**가 더 의미 있음.

---

## 9. CV 점수 vs 리더보드 점수가 다를 때

### 원인 & 대응
| 원인 | 증상 | 해결 |
|------|------|------|
| Train/Test 분포 다름 | LB가 CV보다 많이 낮음 | Adversarial Val로 확인 |
| 시간적 분포 변화 | | TimeSeriesSplit |
| Public LB가 작은 표본 | LB는 불안정, CV 신뢰해야 | CV 우선 |
| Leakage 존재 | CV가 너무 높음 | 피처 로직 재점검 |
| Overfitting to Public LB | LB 점점 올라가는데 CV 안 오름 | LB 튜닝 중단 |

### 🔑 황금률
> **항상 본인의 CV를 신뢰하라. Public LB는 힌트일 뿐.**

---

## 10. Out-of-Fold (OOF) 예측 패턴 — 스태킹 필수

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def get_oof_predictions(model_class, X, y, n_splits=5, **model_kw):
    oof_pred = np.zeros(len(y))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    for fold, (tr, val) in enumerate(skf.split(X, y)):
        m = model_class(**model_kw)
        m.fit(X[tr], y[tr])
        oof_pred[val] = m.predict_proba(X[val])[:, 1]
    return oof_pred

X, y = make_classification(n_samples=500, random_state=0)
oof = get_oof_predictions(RandomForestClassifier, X, y, n_splits=5, n_estimators=100, random_state=0)
print(f"OOF AUC: {roc_auc_score(y, oof):.4f}")
print(f"OOF shape: {oof.shape}")
# oof 는 나중에 메타 모델(스태킹)의 입력으로 사용됨

---

## 📝 Day 15-16 체크리스트
- [ ] KFold / StratifiedKFold 차이 이해
- [ ] GroupKFold 사용 상황 인지
- [ ] TimeSeriesSplit 필요성 이해
- [ ] Adversarial Validation 개념
- [ ] 지표별 용도 구분 가능
- [ ] OOF 예측 직접 구현

다음은 **XGBoost / LightGBM / CatBoost** 로 실전 모델링입니다.